# D2 — Rana — Climate Evidence Knowledge Graph: Build & Cypher Evidence

**Owner:** Rana  
**Course:** CSAI415 — Climate Evidence GraphRAG Agent  
**Deliverable:** D2 — Graph Construction  
**File:** `notebooks/D2_03_Rana_graph_build_cypher.ipynb`

---

## What this notebook proves

| Rubric item | Evidence produced here |
|---|---|
| Neo4j graph schema is defined | Schema loaded and constraints verified in Section 1 |
| Climate-specific nodes/relationships exist | Node counts by label in Section 4 |
| 3–5 Cypher queries return useful evidence | 5 GraphRAG reasoning queries in Section 5 |
| Final graph counts written to CSV | `reports/tables/d2_graph_counts.csv` in Section 6 |

---

## Notebook structure

| Section | Purpose |
|---|---|
| 0. Setup | Install deps, configure logging, set paths |
| 1. Neo4j Connection & Constraints | Connect via `ClimateGraphBuilder`, run constraints |
| 2. Metadata Ingestion | Load CSV → upsert Document + entity nodes |
| 3. Relationship Construction | Metadata-safe relationships + Finding evidence edges |
| 4. Validation Checks | Integrity checks — orphans, missing pages, noisy edges |
| 5. GraphRAG Reasoning Queries | 5 multi-hop Cypher queries with explanations |
| 6. Graph Statistics + CSV Export | Node/rel counts → `d2_graph_counts.csv` |
| 7. Reflection | Why the graph improves over vector-only retrieval |

---
## Section 0 — Setup

Install dependencies, configure logging, and set project paths.
All graph logic is imported from `src/graph/` — no raw Cypher is written in this notebook.
All query strings come from `cypher_queries.py`.

In [7]:
%pip install neo4j pandas tqdm --quiet

Note: you may need to restart the kernel to use updated packages.


In [8]:
import logging
import sys
import os
from pathlib import Path

import pandas as pd
from tqdm import tqdm

# ── Project root resolution ────────────────────────────────────────────────
# Works whether Jupyter starts in the repo root or inside notebooks/.
cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ── Logging configuration ──────────────────────────────────────────────────
# Set to DEBUG to see every MERGE operation. Set to WARNING for quiet runs.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('D2_Rana')

# ── Path constants ─────────────────────────────────────────────────────────
METADATA_PATH = ROOT / 'data' / 'metadata' / 'papers_metadata.csv'
FINDINGS_PATH = ROOT / 'data' / 'metadata' / 'findings_metadata.csv'
REPORTS_DIR = ROOT / 'reports' / 'tables'
GRAPH_COUNTS_PATH = REPORTS_DIR / 'd2_graph_counts.csv'

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

logger.info('ROOT: %s', ROOT)
logger.info('Metadata path exists: %s', METADATA_PATH.exists())
logger.info('Findings path exists: %s', FINDINGS_PATH.exists())

12:00:20 [INFO] D2_Rana: ROOT: C:\Users\Administrator\Documents\GitHub\climate_evidence_graphrag_agent
12:00:20 [INFO] D2_Rana: Metadata path exists: True
12:00:20 [INFO] D2_Rana: Findings path exists: False


---
## Section 1 — Neo4j Connection & Constraint Setup

### Why constraints matter before any write

Neo4j's `MERGE` operation finds or creates a node based on a property key.
Without a **uniqueness constraint** on that key, two concurrent writes of the
same entity can produce two separate nodes — breaking the deduplication guarantee.

For a climate knowledge graph, this matters because:
- `"UAE"` and `"United Arab Emirates"` must resolve to **one** Country node
- `"Carbon Pricing"` and `"carbon_pricing"` must resolve to **one** Policy node
- Two findings from the same page of the same document must not create duplicates

Constraints are idempotent (`IF NOT EXISTS`) — safe to re-run at every notebook start.

### How to run this notebook with Neo4j

Before running the graph cells, create or start a Neo4j instance.

**Option A — Local Neo4j / Docker:**
1. Start Neo4j locally.
2. Use Bolt URI: `bolt://localhost:7687`.
3. Use your Neo4j username and password in the connection cell below.

**Option B — Neo4j Aura / Cloud instance:**
1. Create a new Neo4j Aura instance from the Neo4j website.
2. Copy the generated connection URI. It usually looks like `neo4j+s://...databases.neo4j.io`.
3. Copy the generated username and password.
4. Paste those values in the connection cell below.

**Important:** Do not submit private passwords in screenshots, GitHub commits, or the final report. For the professor to run the notebook, they should create their own Neo4j instance and enter their own username/password.

In [9]:
from graph.neo4j_builder import ClimateGraphBuilder
from graph.cypher_queries import (
    CLIMATE_CYPHER_QUERIES,
    GRAPHRAG_REASONING,
    VALIDATION_QUERIES,
    GRAPH_STATISTICS,
    get_query,
    list_queries,
)

# ── Connection configuration ───────────────────────────────────────────────
# Professor/run instructions:
# 1. Create or start a Neo4j instance.
# 2. Put that instance's URI, username, and password below.
# 3. For local Neo4j, the URI is usually bolt://localhost:7687.
# 4. For Neo4j Aura/cloud, use the generated neo4j+s://... URI.
#
# You can also set environment variables instead of editing this cell:
# NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD.
NEO4J_URI      = os.environ.get('NEO4J_URI',      'bolt://localhost:7687')
NEO4J_USER     = os.environ.get('NEO4J_USER',     'neo4j')
NEO4J_PASSWORD = os.environ.get('NEO4J_PASSWORD', 'PUT_YOUR_NEO4J_PASSWORD_HERE')

if NEO4J_PASSWORD == 'PUT_YOUR_NEO4J_PASSWORD_HERE':
    raise ValueError(
        'Please create/start a Neo4j instance and put your Neo4j password in '
        'NEO4J_PASSWORD, or set the NEO4J_PASSWORD environment variable.'
    )

# ── Initialise builder ────────────────────────────────────────────────────
# ClimateGraphBuilder verifies connectivity on __init__.
# If Neo4j is unavailable, it raises ConnectionError after 3 retries.
builder = ClimateGraphBuilder(
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD,
)
print('ClimateGraphBuilder connected successfully.')

ValueError: Please create/start a Neo4j instance and put your Neo4j password in NEO4J_PASSWORD, or set the NEO4J_PASSWORD environment variable.

In [ ]:
# ── Run constraints and indexes ────────────────────────────────────────────
# This must be called before any upsert operation.
# Constraints created:
#   - doc_id_unique, country_id_unique, policy_id_unique, risk_id_unique,
#     tech_id_unique, finding_id_unique, sector_id_unique, region_id_unique,
#     scenario_id_unique, target_id_unique
# Indexes created:
#   - finding_text_index (fulltext), doc_year_index, risk_type_index,
#     finding_page_index (composite doc_id + page)

builder.run_constraints()
print('Constraints and indexes applied.')

# Verify by listing constraints
from neo4j import GraphDatabase
_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with _driver.session() as session:
    constraints = list(session.run('SHOW CONSTRAINTS'))
    print(f'\nTotal constraints in database: {len(constraints)}')
    for c in constraints:
        print(f"  {c['name']}: {c.get('labelsOrTypes', '')} → {c.get('properties', '')}")
_driver.close()

---
## Section 2 — Metadata Ingestion

### Ingestion pipeline (two-pass design)

**Pass 1 — Nodes first:**  
Every Document, Organization, Venue, Country, Region, Sector, ClimateRisk,
Technology, Policy, and Target node is MERGEd before any relationship is written.
This prevents MATCH failures in Pass 2.

**Pass 2 — Relationships second:**  
Only safe, metadata-derivable relationships are written here:
- `Document -[:DISCUSSES]-> ClimateTopic`
- `Document -[:MENTIONS_COUNTRY]-> Country`
- `Country -[:LOCATED_IN]-> Region`  *(geographic fact — always safe)*
- `Country -[:HAS_POLICY]-> Policy`  *(only for policy/ndc/strategy doc_type)*
- `Policy -[:SETS_TARGET]-> Target`  *(only when both in same document)*

**Causal relationships (IMPACTS, MITIGATES, OCCURS_IN) are written in Section 3.**
They require validated Finding evidence — never metadata co-occurrence.

### Entity normalisation

Before every MERGE, `normalize_entity_name()` resolves aliases:
- `"uae"` → `"United Arab Emirates"`
- `"solar"` → `"Solar PV"`
- `"ccs"` → `"Carbon Capture and Storage"`

Then `slugify()` generates the stable ID: `"united_arab_emirates"`, `"solar_pv"`, etc.
Two name variants that normalise to the same canonical form share one node.

In [ ]:
# ── Load metadata CSV ─────────────────────────────────────────────────────
if not METADATA_PATH.exists():
    logger.warning('papers_metadata.csv not found — using sample data for demonstration.')
    # Sample records representing the UAE climate document corpus
    records = [
        {
            'doc_id': 'uae_netzero_2050',
            'title': 'UAE Net Zero by 2050 Strategic Initiative',
            'year': 2021,
            'doc_type': 'strategy',
            'organization': 'Ministry of Climate Change and Environment',
            'venue': 'UAE Government Publications',
            'language': 'en',
            'page_count': 48,
            'url': 'https://www.moccae.gov.ae',
            'topics': 'Renewable Energy Transition|Climate Governance|Energy Efficiency',
            'countries': 'United Arab Emirates',
            'regions': 'Middle East and North Africa|Gulf Cooperation Council',
            'sectors': 'Renewable Energy|Transport|Buildings|Industry',
            'climate_risks': 'Heatwaves|Water Scarcity|Sea Level Rise',
            'technologies': 'Solar PV|Green Hydrogen|Carbon Capture and Storage',
            'policies': 'UAE Net Zero by 2050|UAE Energy Strategy 2050',
            'targets': 'Net Zero by 2050|44% Clean Energy by 2050|Triple Renewables by 2030',
        },
        {
            'doc_id': 'ipcc_ar6_wg2',
            'title': 'IPCC Sixth Assessment Report - Working Group II: Impacts, Adaptation and Vulnerability',
            'year': 2022,
            'doc_type': 'ipcc_report',
            'organization': 'IPCC',
            'venue': 'IPCC Reports',
            'language': 'en',
            'page_count': 3068,
            'url': 'https://www.ipcc.ch/report/ar6/wg2/',
            'topics': 'Climate Impacts|Adaptation|Loss and Damage|Food Security',
            'countries': 'United Arab Emirates|Saudi Arabia|Egypt|Jordan',
            'regions': 'Middle East and North Africa|Global South',
            'sectors': 'Agriculture|Coastal Infrastructure|Public Health|Water Resources',
            'climate_risks': 'Heatwaves|Water Scarcity|Coastal Flooding|Desertification|Sea Level Rise',
            'technologies': 'Desalination|Renewable Energy|Climate-Smart Agriculture',
            'policies': 'Paris Agreement|UAE Food Security Strategy',
            'targets': '1.5C Warming Limit|2C Warming Limit',
        },
        {
            'doc_id': 'cop28_uae_synthesis',
            'title': 'COP28 UAE Synthesis Report: Global Stocktake Outcomes',
            'year': 2023,
            'doc_type': 'cop_document',
            'organization': 'UNFCCC',
            'venue': 'UNFCCC Publications',
            'language': 'en',
            'page_count': 112,
            'url': 'https://unfccc.int/cop28',
            'topics': 'Mitigation|Climate Finance|Renewable Energy Transition|Loss and Damage',
            'countries': 'United Arab Emirates|Global',
            'regions': 'Global|Middle East and North Africa',
            'sectors': 'Renewable Energy|Industry|Transport|Agriculture',
            'climate_risks': 'Carbon Emissions|Heatwaves|Sea Level Rise',
            'technologies': 'Solar PV|Wind Power|Green Hydrogen|Direct Air Capture|Battery Energy Storage Systems',
            'policies': 'COP28 UAE Agreement|Paris Agreement|Global Stocktake',
            'targets': 'Triple Renewables by 2030|Phase Down Fossil Fuels|Net Zero by 2050',
        },
        {
            'doc_id': 'uae_ndc_2022',
            'title': 'UAE Nationally Determined Contribution 2022 Update',
            'year': 2022,
            'doc_type': 'ndc',
            'organization': 'Ministry of Climate Change and Environment',
            'venue': 'UNFCCC NDC Registry',
            'language': 'en',
            'page_count': 36,
            'url': 'https://unfccc.int/ndc/uae',
            'topics': 'Mitigation|Adaptation|Climate Governance',
            'countries': 'United Arab Emirates',
            'regions': 'Middle East and North Africa|Gulf Cooperation Council',
            'sectors': 'Energy|Transport|Waste|Agriculture',
            'climate_risks': 'Carbon Emissions|Heatwaves|Water Scarcity',
            'technologies': 'Solar PV|Green Hydrogen|Carbon Capture and Storage|Electric Vehicles',
            'policies': 'UAE Nationally Determined Contribution|UAE Energy Strategy 2050',
            'targets': '31% GHG Reduction by 2030|44% Renewable Share by 2050',
        },
        {
            'doc_id': 'ipcc_ar6_wg3',
            'title': 'IPCC Sixth Assessment Report - Working Group III: Mitigation of Climate Change',
            'year': 2022,
            'doc_type': 'ipcc_report',
            'organization': 'IPCC',
            'venue': 'IPCC Reports',
            'language': 'en',
            'page_count': 2258,
            'url': 'https://www.ipcc.ch/report/ar6/wg3/',
            'topics': 'Mitigation|Renewable Energy|Carbon Pricing|Technology Deployment',
            'countries': 'Global',
            'regions': 'Global|Middle East and North Africa',
            'sectors': 'Energy|Industry|Transport|Buildings|Agriculture',
            'climate_risks': 'Carbon Emissions|Warming above 1.5C|Stranded Assets',
            'technologies': 'Solar PV|Wind Power|Green Hydrogen|Direct Air Capture|Carbon Capture and Storage|Electric Vehicles|Battery Energy Storage Systems',
            'policies': 'Paris Agreement|Carbon Pricing|Fossil Fuel Phase-Out',
            'targets': 'Net Zero by 2050|1.5C Warming Limit|60-70% Renewable Electricity by 2030',
        },
    ]
    metadata_df = pd.DataFrame(records)
else:
    metadata_df = pd.read_csv(METADATA_PATH)
    records = metadata_df.to_dict('records')

print(f'Loaded {len(metadata_df)} document records.')
print(f'Columns: {list(metadata_df.columns)}')
metadata_df[['doc_id', 'title', 'year', 'doc_type']].head()

In [ ]:
# ── Metadata audit before ingestion ──────────────────────────────────────
# Flag any records with missing critical fields before writing to Neo4j.
# A missing doc_id will cause the MERGE to fail silently.

critical_fields = ['doc_id', 'title', 'year', 'doc_type']
audit_rows = []

for rec in records:
    issues = [f for f in critical_fields if not rec.get(f)]
    if issues:
        audit_rows.append({'doc_id': rec.get('doc_id', 'MISSING'), 'missing_fields': issues})

if audit_rows:
    print(f'WARNING: {len(audit_rows)} records have missing critical fields:')
    for row in audit_rows:
        print(f"  doc_id={row['doc_id']} missing: {row['missing_fields']}")
else:
    print(f'Audit passed: all {len(records)} records have required fields.')

In [ ]:
# ── Two-pass bulk ingestion ───────────────────────────────────────────────
# Pass 1: All nodes (Document, Organization, Venue, all entity types)
# Pass 2: All safe metadata relationships
#
# bulk_upsert_documents() handles both passes in order.
# Running this cell multiple times is safe — MERGE ensures idempotency.

print('Starting bulk ingestion...')
result = builder.bulk_upsert_documents(records, include_relationships=True)
print(f"\nIngestion complete:")
print(f"  Processed: {result['processed']}")
print(f"  Errors:    {result['errors']}")

if result['errors'] > 0:
    print('\nWARNING: Some records failed. Check log output above for details.')

---
## Section 3 — Finding Evidence Construction

### Why Findings are the most important nodes

Every `Finding` node is a **page-anchored evidence claim** extracted from a PDF.
It is the bridge between:
- The **graph structure** (ClimateRisk, Sector, Country, Technology)
- The **Qdrant vector store** (the actual chunk text at that page)

Without Finding nodes, the graph cannot close the citation loop:
*"This answer is supported by IPCC AR6 WG2, page 1147."*

### What makes a Finding safe to write causal edges from

A Finding must have:
1. A non-null `doc_id` (points to an existing Document)
2. A non-null `page` (enables Qdrant chunk lookup)
3. A non-empty `text` (the evidence statement)
4. At least one entity target (`risk`, `sector`, `country`, or `technology`)

Only then are the `EVIDENCES_RISK`, `EVIDENCES_SECTOR` etc. edges written.
These edges then unlock the causal edges: `IMPACTS`, `MITIGATES`, `OCCURS_IN`.

**Causal edges are only written when both ends are grounded in the same Finding.**
This prevents the graph from asserting that "heatwaves impact agriculture"
simply because both appeared in a metadata list for the same document.

In [ ]:
# ── Load finding records ──────────────────────────────────────────────────
# In production: load from findings_metadata.csv (output of PDF extraction pipeline)
# Here: curated sample findings with full evidence provenance

if FINDINGS_PATH.exists():
    findings_df = pd.read_csv(FINDINGS_PATH)
    finding_records = findings_df.to_dict('records')
    print(f'Loaded {len(finding_records)} findings from {FINDINGS_PATH}')
else:
    print('findings_metadata.csv not found — using curated sample findings.')
    finding_records = [
        {
            'doc_id': 'ipcc_ar6_wg2',
            'page': 1147,
            'text': 'Sea level rise poses significant threats to coastal infrastructure in MENA, '
                    'with projections of 0.5-1.0m rise by 2100 under high emissions scenarios, '
                    'threatening coastal cities including Abu Dhabi and Dubai.',
            'confidence': 'high',
            'extraction_method': 'manual',
            'risk': 'Sea Level Rise',
            'sector': 'Coastal Infrastructure',
            'country_id': 'ARE',
            'technology': '',
            'qdrant_chunk_id': 'ipcc_ar6_wg2_chunk_1147_01',
        },
        {
            'doc_id': 'ipcc_ar6_wg2',
            'page': 1089,
            'text': 'Heatwave frequency and intensity have increased significantly across the MENA region, '
                    'with wet-bulb temperatures in Gulf countries approaching the limits of human '
                    'physiological tolerance under RCP8.5 scenarios. Agricultural yields in '
                    'the region are projected to decline 20-40% by 2050.',
            'confidence': 'high',
            'extraction_method': 'manual',
            'risk': 'Heatwaves',
            'sector': 'Agriculture',
            'country_id': 'ARE',
            'technology': '',
            'qdrant_chunk_id': 'ipcc_ar6_wg2_chunk_1089_01',
        },
        {
            'doc_id': 'ipcc_ar6_wg3',
            'page': 624,
            'text': 'Green hydrogen produced from renewable electricity can effectively reduce '
                    'carbon emissions in hard-to-abate industrial sectors. Cost reductions of '
                    '50-80% are projected by 2030 under rapid deployment scenarios.',
            'confidence': 'medium',
            'extraction_method': 'manual',
            'risk': 'Carbon Emissions',
            'sector': 'Industry',
            'country_id': '',
            'technology': 'Green Hydrogen',
            'qdrant_chunk_id': 'ipcc_ar6_wg3_chunk_624_01',
        },
        {
            'doc_id': 'ipcc_ar6_wg3',
            'page': 388,
            'text': 'Solar PV has demonstrated consistent and rapid cost reductions, with '
                    'Levelised Cost of Electricity below USD 0.03/kWh in high-irradiance '
                    'regions including the UAE and Saudi Arabia, making it the cheapest '
                    'source of electricity generation in history and a primary mitigation option '
                    'for carbon emissions from the energy sector.',
            'confidence': 'high',
            'extraction_method': 'manual',
            'risk': 'Carbon Emissions',
            'sector': 'Renewable Energy',
            'country_id': 'ARE',
            'technology': 'Solar PV',
            'qdrant_chunk_id': 'ipcc_ar6_wg3_chunk_388_01',
        },
        {
            'doc_id': 'ipcc_ar6_wg2',
            'page': 1203,
            'text': 'Water scarcity is projected to intensify across the MENA region under '
                    'all warming scenarios, with annual freshwater availability declining '
                    '20-30% by mid-century. This directly threatens food production, '
                    'energy generation, and public health infrastructure in GCC countries.',
            'confidence': 'high',
            'extraction_method': 'manual',
            'risk': 'Water Scarcity',
            'sector': 'Water Resources',
            'country_id': 'ARE',
            'technology': '',
            'qdrant_chunk_id': 'ipcc_ar6_wg2_chunk_1203_01',
        },
        {
            'doc_id': 'cop28_uae_synthesis',
            'page': 34,
            'text': 'Direct Air Capture technology, while currently at low TRL and high cost, '
                    'has potential to remove legacy carbon emissions and contribute to '
                    'net-zero pathways. COP28 pledges totalling USD 4 billion were directed '
                    'toward carbon removal innovation including DAC.',
            'confidence': 'medium',
            'extraction_method': 'llm_extracted',
            'risk': 'Carbon Emissions',
            'sector': 'Industry',
            'country_id': '',
            'technology': 'Direct Air Capture',
            'qdrant_chunk_id': 'cop28_uae_synthesis_chunk_34_01',
        },
    ]
    print(f'Using {len(finding_records)} curated sample findings.')

pd.DataFrame(finding_records)[['doc_id', 'page', 'risk', 'sector', 'technology', 'confidence']]

In [ ]:
# ── Ingest findings and causal relationships ──────────────────────────────
# bulk_upsert_findings() performs two operations per finding:
#   1. upsert_finding() — MERGE the Finding node with page + text
#   2. upsert_finding_relationships() — write EVIDENCES_* and causal edges
#
# Causal edges written:
#   - ClimateRisk -[:IMPACTS {confidence, source}]-> Sector
#   - Technology -[:MITIGATES {confidence, source}]-> ClimateRisk
#   - ClimateRisk -[:OCCURS_IN {confidence, source}]-> Region (via Country)

print('Ingesting findings and causal relationships...')
finding_result = builder.bulk_upsert_findings(finding_records)
print(f"\nFinding ingestion complete:")
print(f"  Processed: {finding_result['processed']}")
print(f"  Errors:    {finding_result['errors']}")

---
## Section 4 — Validation Checks

A graph can load without errors and still be unusable for GraphRAG.
The following checks detect the most common failure modes:

| Check | What failure means |
|---|---|
| Findings missing page | Qdrant chunk lookup will fail — these findings are useless |
| Orphan Findings | Finding exists but has no entity edges — graph dead-end |
| Findings not linked to Document | Graph provenance chain broken |
| Isolated ClimateRisk nodes | Risk exists but cannot be traversed to or from |
| High-cardinality Policy nodes | Metadata co-mention noise — policy connected to too many countries |
| Causal edges from metadata source | Safety gate bypassed — edges without Finding evidence |

In [ ]:
# ── Finding integrity validation ──────────────────────────────────────────
print('Running Finding integrity validation...')
integrity = builder.validate_finding_integrity()

integrity_df = pd.DataFrame([
    {'check': 'total_findings',       'count': integrity['total_findings'],       'status': 'INFO'},
    {'check': 'missing_page',         'count': integrity['missing_page'],         'status': 'FAIL' if integrity['missing_page'] > 0 else 'PASS'},
    {'check': 'missing_doc_id',       'count': integrity['missing_doc_id'],       'status': 'FAIL' if integrity['missing_doc_id'] > 0 else 'PASS'},
    {'check': 'no_entity_edges',      'count': integrity['no_entity_edges'],      'status': 'WARN' if integrity['no_entity_edges'] > 0 else 'PASS'},
    {'check': 'not_linked_to_document','count': integrity['not_linked_to_document'],'status': 'FAIL' if integrity['not_linked_to_document'] > 0 else 'PASS'},
])

print('\nFinding Integrity Report:')
print(integrity_df.to_string(index=False))

# Hard fail on critical issues
critical_failures = integrity_df[integrity_df['status'] == 'FAIL']
if not critical_failures.empty:
    print(f'\n{len(critical_failures)} CRITICAL FAILURES detected. Fix before proceeding to GraphRAG queries.')
else:
    print('\nAll critical checks PASSED.')

In [ ]:
# ── Orphan node check ─────────────────────────────────────────────────────
print('Checking for orphan nodes (nodes with zero relationships)...')
orphans = builder.get_orphan_counts()

if orphans:
    print('\nOrphan nodes detected:')
    orphan_df = pd.DataFrame(orphans)
    print(orphan_df.to_string(index=False))
    print('\nReason: These nodes were created from metadata but never connected.')
    print('Action: Investigate which metadata fields produced orphan entities.')
else:
    print('No orphan nodes. Graph is fully connected.')

In [ ]:
# ── Run named validation queries from cypher_queries.py ──────────────────
from neo4j import GraphDatabase

_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

validation_checks = [
    ('validation_findings_missing_page',    'Findings missing page number (CRITICAL)'),
    ('validation_isolated_risks',           'Isolated ClimateRisk nodes (no connections)'),
    ('validation_high_cardinality_policy',  'Policies with >10 country connections (noise check)'),
    ('validation_metadata_causal_edges',    'Causal edges with source=metadata (safety gate check)'),
]

print('Validation Query Results:')
print('=' * 60)

for query_name, description in validation_checks:
    print(f'\n{description}')
    print('-' * 40)
    with _driver.session() as session:
        results = list(session.run(get_query(query_name)))
    if results:
        df = pd.DataFrame([dict(r) for r in results])
        print(df.to_string(index=False))
    else:
        print('  ✓ No issues found.')

_driver.close()

---
## Section 5 — GraphRAG Reasoning Queries

These five queries demonstrate what the Climate Evidence Knowledge Graph
enables beyond vector-only retrieval. Each query implements a multi-hop
reasoning path that a semantic search over document chunks cannot replicate.

In the D3 GraphRAG pipeline:
1. A user question is parsed to extract named entities
2. The appropriate GraphRAG query is selected and executed
3. The `doc_id + page` pairs from the result are used as a constrained
   candidate set for Qdrant dense retrieval
4. The retrieved chunks are assembled into a cited, evidence-grounded answer

The `qdrant_chunk_id` field in the output is the direct bridge to Qdrant.

In [ ]:
# ── Helper: run and display a GraphRAG query ──────────────────────────────
_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_graphrag_query(query_name: str, params: dict, title: str) -> pd.DataFrame:
    """Execute a named GraphRAG query and return results as a DataFrame."""
    cypher = get_query(query_name)
    with _driver.session() as session:
        results = list(session.run(cypher, params))
    if results:
        df = pd.DataFrame([dict(r) for r in results])
        print(f'\n{title}')
        print(f'Rows returned: {len(df)}')
        return df
    else:
        print(f'\n{title}: No results (graph may need more data ingested).')
        return pd.DataFrame()

### GraphRAG Query 1: Country → Policy → Target

**Question:** *What climate targets has the UAE formally committed to, under which adopted policies, and where is this documented?*

**Why vector search cannot answer this alone:**  
A dense retrieval query for "UAE renewable energy target 2050" retrieves chunks
co-mentioning these terms — but cannot enforce that:
- UAE is the **committing nation** (not just mentioned in context)
- The policy is **adopted** (not proposed or expired)
- The target belongs to that specific policy (not from another country's NDC in the same doc)

**Traversal:** `Country(ARE) -HAS_POLICY→ Policy(status=adopted) -SETS_TARGET→ Target`  
+ OPTIONAL JOIN to Finding evidence with page number for Qdrant retrieval.

In [ ]:
q1 = run_graphrag_query(
    'graphrag_country_policy_target',
    params={'country_id': 'ARE'},
    title='GRAPHRAG Q1: UAE Country → Policy → Target Chain',
)
if not q1.empty:
    display_cols = [c for c in ['country','policy','target','target_value','deadline','source_doc','evidence_page'] if c in q1.columns]
    display(q1[display_cols])

### GraphRAG Query 2: Policy → ClimateRisk → Sector

**Question:** *Which climate risks and sectors does the UAE Net Zero strategy address, and what is the evidence confidence?*

**Why vector search cannot answer this alone:**  
Semantic search for "UAE Net Zero policy sectors" retrieves co-mentioned content but
cannot distinguish whether the risk is one the policy **addresses** (via ADDRESSES edge)
or merely one the document mentions in passing. It also cannot rank by evidence confidence.

**Traversal:** `Policy -ADDRESSES→ ClimateTopic -HAS_RISK→ ClimateRisk -IMPACTS(confidence=high/medium)→ Sector`  
+ OPTIONAL JOIN to Finding evidence for page-anchored retrieval.

In [ ]:
from graph.neo4j_builder import slugify

q2 = run_graphrag_query(
    'graphrag_policy_risk_sector',
    params={'policy_id': slugify('UAE Net Zero by 2050')},
    title='GRAPHRAG Q2: Policy → ClimateRisk → Sector',
)
if not q2.empty:
    display_cols = [c for c in ['policy','climate_topic','climate_risk','affected_sector','impact_confidence','evidence_page','source_doc'] if c in q2.columns]
    display(q2[display_cols])

### GraphRAG Query 3: Technology → MITIGATES → ClimateRisk

**Question:** *Which technologies mitigate which climate risks, with what evidence confidence, and in which sectors and regions?*

**Why vector search cannot answer this alone:**  
A query for "green hydrogen carbon emissions" retrieves chunks where both appear —
but cannot filter to only passages where mitigation is **asserted** vs. mentioned speculatively.
It cannot rank by IPCC confidence, connect to deployment sectors, or chain to regional relevance.

**Traversal:** `Technology -MITIGATES(confidence=high/medium)→ ClimateRisk`  
+ OPTIONAL MATCH `Technology -USED_IN→ Sector`  
+ OPTIONAL MATCH `ClimateRisk -OCCURS_IN→ Region`  
+ OPTIONAL JOIN to Finding evidence for Qdrant chunk delivery.

In [ ]:
q3 = run_graphrag_query(
    'graphrag_technology_mitigates_risk',
    params={'tech_id': None},  # None = return all technologies
    title='GRAPHRAG Q3: Technology → MITIGATES → ClimateRisk (all technologies)',
)
if not q3.empty:
    display_cols = [c for c in ['technology','mitigated_risk','mitigation_confidence','applicable_sector','relevant_region','evidence_page','source_doc','source_year'] if c in q3.columns]
    display(q3[display_cols])

### GraphRAG Query 4: Finding → Document (Evidence Grounding)

**Question:** *What are all page-anchored evidence statements about heatwave impacts, ranked by confidence, with Qdrant chunk IDs for text retrieval?*

**Why vector search cannot answer this alone:**  
Vector search returns chunks ranked by similarity — not by whether the chunk contains
a **validated evidence claim**. The graph's Finding nodes are pre-validated evidence
statements with known confidence levels and exact page locations.
The `qdrant_chunk_id` field is the direct bridge from graph to vector store.

**Traversal:** `Finding -EVIDENCES_RISK→ ClimateRisk(risk_id='heatwaves')`  
+ MATCH `Document -REPORTS_FINDING→ Finding`  
+ OPTIONAL MATCH `Document -PUBLISHED_BY→ Organization`  
Returns ranked by confidence DESC, year DESC.

In [ ]:
q4 = run_graphrag_query(
    'graphrag_finding_document_grounding',
    params={
        'risk_id':          slugify('Heatwaves'),
        'sector_id':        None,
        'country_id':       None,
        'tech_id':          None,
        'min_confidence':   'high',
        'confidence_levels': ['high', 'medium'],
    },
    title='GRAPHRAG Q4: Finding → Document Evidence Grounding (Heatwaves, high+medium confidence)',
)
if not q4.empty:
    display_cols = [c for c in ['evidence_text','page','confidence','qdrant_chunk_id','doc_id','doc_type','publisher','doc_year'] if c in q4.columns]
    display(q4[display_cols])

### GraphRAG Query 5: Region → ClimateRisk

**Question:** *What are the high-confidence climate risks in the MENA region, which sectors do they impact, and which country policies address them?*

**Why vector search cannot answer this alone:**  
Semantic search for "MENA climate risks" retrieves documents where MENA and risks
are co-mentioned — but cannot distinguish risks that **occur in** MENA from risks
merely discussed in a MENA-focused document. It cannot chain from regional risk
to sector impact to relevant national policies in a single structured path.

**Traversal:** `Region(mena) ←OCCURS_IN(confidence=high/medium)— ClimateRisk`  
+ OPTIONAL MATCH `ClimateRisk -IMPACTS→ Sector`  
+ OPTIONAL MATCH `Country -LOCATED_IN→ Region`  
+ OPTIONAL MATCH `Country -HAS_POLICY→ Policy -ADDRESSES→ Topic -HAS_RISK→ ClimateRisk`  
+ OPTIONAL JOIN to Finding evidence for page-anchored Qdrant retrieval.

In [ ]:
q5 = run_graphrag_query(
    'graphrag_region_climate_risk',
    params={'region_id': slugify('Middle East and North Africa')},
    title='GRAPHRAG Q5: Region (MENA) → ClimateRisk → Sector → Policy',
)
if not q5.empty:
    display_cols = [c for c in ['region','climate_risk','impacted_sector','risk_occurrence_confidence','country','relevant_policy','evidence_page','source_doc'] if c in q5.columns]
    display(q5[display_cols])

_driver.close()

---
## Section 6 — Graph Statistics & CSV Export

This section computes aggregate counts and saves `d2_graph_counts.csv`.
This file is the D2 deliverable for the rubric item:
*"Neo4j graph: Node/edge counts, 3–5 Cypher query outputs, graph diagram or schema."*

In [ ]:
# ── Node counts by label ──────────────────────────────────────────────────
print('Computing node counts by label...')
node_counts = builder.get_node_counts()

node_counts_df = pd.DataFrame(
    [{'type': 'node', 'label_or_relationship': k, 'count': v, 'category': 'node'}
     for k, v in node_counts.items()]
)
print('\nNode counts:')
display(node_counts_df)

In [ ]:
# ── Relationship counts by type ───────────────────────────────────────────
print('Computing relationship counts...')
rel_counts = builder.get_relationship_counts()

rel_counts_df = pd.DataFrame(
    [{'type': 'relationship', 'label_or_relationship': k, 'count': v, 'category': 'relationship'}
     for k, v in rel_counts.items()]
)
print('\nRelationship counts:')
display(rel_counts_df)

In [ ]:
# ── Additional statistics ─────────────────────────────────────────────────
_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

stat_queries = [
    ('stats_graph_density',          'Graph density (avg rels per node)'),
    ('stats_finding_coverage',       'Finding coverage across documents'),
    ('stats_technology_coverage',    'Technology mitigation coverage'),
    ('stats_risk_coverage',          'ClimateRisk impact coverage'),
    ('stats_country_policy_coverage','Country policy coverage'),
]

print('Graph Statistics:')
print('=' * 60)
stats_rows = []

for query_name, description in stat_queries:
    with _driver.session() as session:
        results = list(session.run(get_query(query_name)))
    if results:
        r = dict(results[0])
        print(f'\n{description}:')
        for k, v in r.items():
            val_str = f'{v:.2f}' if isinstance(v, float) else str(v)
            print(f'  {k}: {val_str}')
            stats_rows.append({'type': 'statistic', 'label_or_relationship': f'{query_name}.{k}', 'count': v, 'category': description})

_driver.close()

In [ ]:
# ── Assemble and save d2_graph_counts.csv ─────────────────────────────────
graph_counts_df = pd.concat([
    node_counts_df,
    rel_counts_df,
    pd.DataFrame(stats_rows),
], ignore_index=True)

graph_counts_df.to_csv(GRAPH_COUNTS_PATH, index=False)
print(f'd2_graph_counts.csv saved to: {GRAPH_COUNTS_PATH}')
print(f'Total rows: {len(graph_counts_df)}')
display(graph_counts_df.head(30))

---
## Section 7 — Reflection

### Q1: Why does this graph help beyond vector search?

Vector search answers "what chunks are semantically similar to this query?"  
The graph answers "what is structurally true about climate relationships, and where is the evidence?"

Three concrete improvements over vector-only retrieval:

**1. Compound query precision.** A question like "Which technologies mitigate risks that threaten UAE agriculture under adopted policies?" requires four entity types and four relationship hops. Vector search must find a single chunk containing all four — almost impossible. The graph traverses this path exactly.

**2. Evidence confidence filtering.** IPCC AR6 findings marked `confidence=high` are not the same as speculative research paper claims marked `confidence=low`. The graph stores confidence on every causal relationship, enabling retrieval that prioritises authoritative evidence. Vector search has no equivalent signal.

**3. Deterministic citation.** Every answer can be cited to a specific `doc_id + page`, because every Finding node stores this provenance. The `qdrant_chunk_id` field provides the direct bridge to the Qdrant chunk for text delivery. Vector search returns chunks but cannot guarantee the chunk contains the actual evidence claim.

---

### Q2: Which relationship type is most important for climate reasoning?

`EVIDENCES_RISK` — the edge from Finding to ClimateRisk — is the most critical.  
It is the joint that connects the graph's semantic structure to page-anchored evidence.
Without it, `IMPACTS`, `MITIGATES`, and `OCCURS_IN` edges have no grounding,
and the graph is no better than a metadata index.

Second most important: `REPORTS_FINDING` — the Document-to-Finding edge
that enables the Qdrant chunk lookup. If this edge is missing, the hybrid retrieval
system cannot close the loop from graph path to cited text.

---

### Q3: Where is the graph currently weak or incomplete?

Three current limitations:

**1. Finding coverage is low.** The sample data contains 6 manually curated findings.
A production system needs hundreds of extracted findings per document, produced by
the LLM extraction pipeline (D2 Reem's chunk pipeline → D3 extraction step).
Until finding coverage improves, the GraphRAG queries return thin results.

**2. Cascade risk edges (LEADS_TO) are not yet populated.**
The schema defines `ClimateRisk -[:LEADS_TO]-> ClimateRisk` for compound risk chains
(heatwaves → water scarcity → desertification). These edges require multi-hop extraction
from IPCC AR6 chapters — not implemented in D2 metadata ingestion.

**3. Scenario nodes are missing.**
Findings in the sample data do not yet carry SSP scenario context.
Without Scenario nodes, quantitative targets ("44% GHG reduction by 2030")
cannot be contextualised by the warming pathway they assume.
This limits the system's ability to answer scenario-conditional questions.

---

### Q4: Which graph paths are most useful for D3 GraphRAG?

For the D3 GraphRAG agent, the three highest-value traversal paths are:

1. **Evidence retrieval:** `ClimateRisk ←EVIDENCES_RISK— Finding -REPORTS_FINDING← Document`  
   Used to retrieve page-anchored evidence for any risk-related question.  
   Bridge to Qdrant: `Finding.qdrant_chunk_id`

2. **Policy accountability:** `Country -HAS_POLICY→ Policy -SETS_TARGET→ Target ←ASSUMED_SCENARIO— Scenario`  
   Used for NDC transparency and Global Stocktake questions.

3. **Technology selection:** `ClimateRisk ←MITIGATES— Technology -USED_IN→ Sector`  
   Combined with `Technology.trl` for technology readiness filtering —  
   enables questions like "What mature technologies are available to mitigate heatwave risk in UAE agriculture?"

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────────────
builder.close()
print('ClimateGraphBuilder closed.')
print('D2_03_Rana_graph_build_cypher.ipynb complete.')
print(f'Graph counts written to: {GRAPH_COUNTS_PATH}')
print('\nAvailable GraphRAG queries for D3 integration:')
for q in list_queries('graphrag'):
    print(f'  - {q}')